# Transposition Attacks

### Imports

In [1]:
# Core utilities for working with classical ciphers
import random
import math
import itertools
from collections import Counter, defaultdict
import re

### Helper: Text normalization
Transposition cryptanalysis typically removes non-letters and uppercases for consistent scoring and grid handling

In [2]:
ALPHABET = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"

def normalize(text):
    return re.sub('[^A-Z]', '', text.upper())


## Let's start by reviewing some of the tranposition techniques seen in class

### Rail fence cipher (review)
Rail fence writes plaintext on “rails” in a zigzag pattern and reads off row-wise; it is a simple transposition useful to show frequency preservation and structural clues.
Breaking rail fence is often done by trying a small number of rails and reconstructing the zigzag positions, making it a good first exercise in transposition analysis.



In [3]:
def rail_fence_encrypt(pt, rails=3):
    pt = normalize(pt)
    fence = [['\n'] * len(pt) for _ in range(rails)]
    row, step = 0, 1
    for col, ch in enumerate(pt):
        fence[row][col] = ch
        if row == 0:
            step = 1
        elif row == rails - 1:
            step = -1
        row += step
    ct = ''.join(ch for r in fence for ch in r if ch != '\n')
    return ct

def rail_fence_decrypt(ct, rails=3):
    ct = normalize(ct)
    fence = [['\n'] * len(ct) for _ in range(rails)]
    # Mark zigzag path
    row, step = 0, 1
    for col in range(len(ct)):
        fence[row][col] = '*'
        if row == 0:
            step = 1
        elif row == rails - 1:
            step = -1
        row += step
    # Fill row-wise
    idx = 0
    for r in range(rails):
        for c in range(len(ct)):
            if fence[r][c] == '*' and idx < len(ct):
                fence[r][c] = ct[idx]
                idx += 1
    # Read zigzag
    row, step = 0, 1
    pt = []
    for col in range(len(ct)):
        pt.append(fence[row][col])
        if row == 0:
            step = 1
        elif row == rails - 1:
            step = -1
        row += step
    return ''.join(pt)


In [ ]:
#use rail_fence_encrypt and rail_fence_decrypt functions for testing
if __name__ == "__main__":
    plaintext = "WEAREDISCOVEREDFLEEATONCE"
    rails = 3
    ciphertext = rail_fence_encrypt(plaintext, rails) # Encrypt the plaintext
    print(f"Ciphertext: {ciphertext}")
    decrypted_text = rail_fence_decrypt(ciphertext, rails) # Decrypt the ciphertext
    print(f"Decrypted Text: {decrypted_text}")

Ciphertext: WECRLTEERDSOEEFEAOCAIVDEN
Decrypted Text: WEAREDISCOVEREDFLEEATONCE


Open questions

What observable patterns in rail fence ciphertext reveal likely rail counts or zigzag structure, and how can frequency preservation help distinguish transposition from substitution ?

Can you use IoC or Chi Squares to get to the key? Why? What IoC score are you getting?

Implement an automatic bruteforce breaker using AI that tries all rails from 2 to 10 and try to come with a ranking system that will give you a language score



### Route cipher (review)
Route ciphers write plaintext into a grid and read off using a key-specified path (e.g., down columns, spiral, zigzag), which yields many possible keys, near infinite but also routes that can leave conspicuous plaintext chunks if poorly chosen.

Cryptanalysis probes plausible grid dimensions and routes, often using multiple anagramming across aligned messages or scoring-based searches to detect likely reading paths.

In [5]:
def route_cipher_encrypt(pt, rows, cols, route='down_columns'):
    pt = normalize(pt)
    if rows * cols < len(pt):
        raise ValueError("Grid too small")
    grid = [['X'] * cols for _ in range(rows)]
    idx = 0
    for r in range(rows):
        for c in range(cols):
            grid[r][c] = pt[idx] if idx < len(pt) else 'X'
            idx += 1
    # Simple example route: read down columns left to right
    out = []
    if route == 'down_columns':
        for c in range(cols):
            for r in range(rows):
                out.append(grid[r][c])
    else:
        raise NotImplementedError("Only 'down_columns' demo route implemented")
    return ''.join(out)

def route_cipher_decrypt(ct, rows, cols, route='down_columns'):
    ct = normalize(ct)
    if len(ct) != rows * cols:
        raise ValueError("Ciphertext length must equal rows*cols for this simple demo")
    grid = [[''] * cols for _ in range(rows)]
    idx = 0
    if route == 'down_columns':
        for c in range(cols):
            for r in range(rows):
                grid[r][c] = ct[idx]
                idx += 1
    else:
        raise NotImplementedError("Only 'down_columns' demo route implemented")
    pt = []
    for r in range(rows):
        for c in range(cols):
            pt.append(grid[r][c])
    return ''.join(pt)


In [6]:
#use toute cipher above
if __name__ == "__main__":
    plaintext = "WEAREDISCOVEREDFLEEATONCE"
    rows, cols = 5, 5
    ciphertext = route_cipher_encrypt(plaintext, rows, cols) # Encrypt the plaintext
    print(f"Route Cipher Ciphertext: {ciphertext}")
    decrypted_text = route_cipher_decrypt(ciphertext, rows, cols) # Decrypt the ciphertext
    print(f"Route Cipher Decrypted Text: {decrypted_text}")

Route Cipher Ciphertext: WDVFTEIELOASRENRCEECEODAE
Route Cipher Decrypted Text: WEAREDISCOVEREDFLEEATONCE


### Cryptanalysis Technicques for Anagramming

In [7]:
# ##############################################################################
# TECHNIQUE 1: SCORING-BASED SEARCH OVER ROUTES
# ##############################################################################
#
# EXPLANATION:
# When we don't know the route, we need a way to programmatically score a decrypted
# plaintext to see how "English-like" it is. A simple and effective method is to use
# N-gram frequencies. We pre-compute the frequencies of common letter groups (like
# quadgrams, e.g., 'TION', 'OUGH') from a large English text. The decrypted text that
# contains more of these common N-grams gets a higher score. We use the logarithm
# of the frequencies to make the math easier and more stable.

# A pre-computed dictionary of English quadgram log-probabilities.
# In a real scenario, this would be generated from a very large text corpus.
QUADGRAM_LOG_PROBS = {
    'TION': -2.3, 'OTHE': -2.5, 'THER': -2.5, 'THAT': -2.6, 'OFTH': -2.7,
    'FTHE': -2.7, 'INGT': -2.8, 'SAND': -2.9, 'TING': -2.9, 'THEC': -3.0,
    'ATIO': -3.0, 'ENTH': -3.1, 'THIS': -3.1, 'FROM': -3.2, 'THEM': -3.2,
}
# A default low score for quadgrams not in our dictionary
UNSEEN_QUADGRAM_SCORE = -10.0

def score_text(text):
    """Scores a text based on its quadgram frequencies."""
    score = 0
    for i in range(len(text) - 3):
        quadgram = text[i:i+4]
        score += QUADGRAM_LOG_PROBS.get(quadgram, UNSEEN_QUADGRAM_SCORE)
    return score

if __name__ == "__main__":
    print("\n" + "#" * 70)
    print("# TECHNIQUE 1: SCORING-BASED SEARCH")
    print("#" * 70)

    # --- Base Code for Students ---
    correct_plaintext = "WEAREDISCOVEREDFLEEATONCE"
    scrambled_text = "EWDSCORVADEEAEEIFCTOLENER" # Same letters, wrong order

    score_correct = score_text(correct_plaintext)
    score_scrambled = score_text(scrambled_text)

    print(f"Score for correct plaintext '{correct_plaintext[:10]}...': {score_correct:.2f}")
    print(f"Score for scrambled text '{scrambled_text[:10]}...': {score_scrambled:.2f}")
    print("\nNOTICE: The more English-like text has a significantly higher (less negative) score.")



######################################################################
# TECHNIQUE 1: SCORING-BASED SEARCH
######################################################################
Score for correct plaintext 'WEAREDISCO...': -220.00
Score for scrambled text 'EWDSCORVAD...': -220.00

NOTICE: The more English-like text has a significantly higher (less negative) score.


Try creating your own sentences and see how their scores compare. Create your own dictionary of digrams, trigrams, quadgrams and cribs
Test it against the route cipher and see how different are the scores based on your knowledge of English

Does a grammatically correct sentence score higher than a random jumble of words?

In [8]:

# ##############################################################################
# TECHNIQUE 2: PRUNING THE ROUTE SPACE
# ##############################################################################
#
# EXPLANATION:
# The number of possible routes is astronomical. We can't test them all. Instead, we
# create a "library" of common, plausible routes that a human might use (e.g., simple
# scans, snakes, spirals). We can then write a brute-force function that tries ONLY
# these routes, scores each result, and presents the one with the highest fitness score.
# This prunes the search space from trillions of possibilities to a handful.

def generate_plausible_routes(rows, cols):
    """Generates a list of common, plausible routes for a given grid size."""
    routes = {}
    
    # Route 1: Down Columns (the one from the original code)
    route_down_cols = []
    for c in range(cols):
        for r in range(rows):
            route_down_cols.append((r, c))
    routes['down_columns'] = route_down_cols

    # Route 2: Right Rows (standard reading)
    route_right_rows = []
    for r in range(rows):
        for c in range(cols):
            route_right_rows.append((r, c))
    routes['right_rows'] = route_right_rows
    
    # Route 3: Down Columns Snake
    route_snake_cols = []
    for c in range(cols):
        if c % 2 == 0: # Even columns go down
            for r in range(rows): route_snake_cols.append((r,c))
        else: # Odd columns go up
            for r in range(rows - 1, -1, -1): route_snake_cols.append((r,c))
    routes['down_columns_snake'] = route_snake_cols

    # Add more routes here for the student activity!
            
    return routes

def route_cipher_decrypt_with_route(ct, rows, cols, route):
    """A more flexible decryptor that uses a list of (r,c) coordinates."""
    grid = [[''] * cols for _ in range(rows)]
    for i, (r, c) in enumerate(route):
        grid[r][c] = ct[i]
    
    # Read out normally to get plaintext
    return ''.join(grid[r][c] for r in range(rows) for c in range(cols))

def brute_force_route_attack(ct, rows, cols):
    """Tries all plausible routes and returns the best-scoring result."""
    plausible_routes = generate_plausible_routes(rows, cols)
    best_score = -float('inf')
    best_plaintext = ""
    best_route_name = ""

    for name, route in plausible_routes.items():
        plaintext_candidate = route_cipher_decrypt_with_route(ct, rows, cols, route)
        current_score = score_text(plaintext_candidate)
        
        print(f"Trying route '{name}'... Score: {current_score:.2f}, PT: '{plaintext_candidate[:15]}...'")
        
        if current_score > best_score:
            best_score = current_score
            best_plaintext = plaintext_candidate
            best_route_name = name
            
    return best_route_name, best_plaintext, best_score


if __name__ == "__main__":
    print("\n" + "#" * 70)
    print("# TECHNIQUE 2: PRUNING THE ROUTE SPACE")
    print("#" * 70)

    # --- Base Code for Students ---
    # We use the same ciphertext from the start
    ciphertext = "WRSCEEEOAFDEIVREEALNXDTCXOE" # "WEAREDISCOVEREDFLEEATONCEXXXX"
    rows, cols = 5, 5
    
    best_route, best_pt, best_sc = brute_force_route_attack(ciphertext, rows, cols)
    
    print("\n--- Brute-Force Attack Results ---")
    print(f"Best route found: {best_route}")
    print(f"Best score: {best_sc:.2f}")
    print(f"Decrypted plaintext: {best_pt}")



######################################################################
# TECHNIQUE 2: PRUNING THE ROUTE SPACE
######################################################################
Trying route 'down_columns'... Score: -220.00, PT: 'WEDEXREEEDSOIAT...'
Trying route 'right_rows'... Score: -220.00, PT: 'WRSCEEEOAFDEIVR...'
Trying route 'down_columns_snake'... Score: -220.00, PT: 'WFDNXRAELDSOIAT...'

--- Brute-Force Attack Results ---
Best route found: down_columns
Best score: -220.00
Decrypted plaintext: WEDEXREEEDSOIATCAVLCEFRNX


Can the previous attack can score higher on different routes? 

What if you encrypt a new message with your new route? Can the attack find it?

In [ ]:
# ##############################################################################
# TECHNIQUE 3: MULTIPLE ANAGRAMMING (DEPTH ATTACK)
# ##############################################################################
#
# EXPLANATION:
# If you have multiple ciphertexts encrypted with the EXACT same key (grid size and
# route), you can perform a powerful "depth" attack. The key insight is that the
# characters at the same position in each ciphertext (e.g., all characters at index 0)
# came from the SAME grid cell.
# By comparing the letter pairings between two columns of ciphertext characters, we
# can guess if their original grid cells were adjacent. For example, if we pair
# column 0 with column 5 and the resulting bigrams ('Q'-'U', 'T'-'H', 'E'-'A') have
# a high English-like score, it's likely grid cell 0 and grid cell 5 were next
# to each other in the original plaintext grid.

# Pre-computed English bigram log-probabilities for scoring
BIGRAM_LOG_PROBS = {
    'TH': -1.5, 'HE': -1.6, 'IN': -1.8, 'ER': -1.9, 'AN': -2.0, 'RE': -2.1,
    'ES': -2.2, 'ON': -2.2, 'ST': -2.3, 'NT': -2.4, 'EN': -2.4, 'AT': -2.5,
}
UNSEEN_BIGRAM_SCORE = -8.0

def score_adjacency(col1_chars, col2_chars):
    """Scores how likely two columns of characters were adjacent."""
    score = 0
    for i in range(len(col1_chars)):
        bigram = col1_chars[i] + col2_chars[i]
        score += BIGRAM_LOG_PROBS.get(bigram, UNSEEN_BIGRAM_SCORE)
    return score

if __name__ == "__main__":
    print("\n" + "#" * 70)
    print("# TECHNIQUE 3: MULTIPLE ANAGRAMMING (DEPTH ATTACK)")
    print("#" * 70)
    
    # --- Base Code for Students ---
    # We need multiple plaintexts of the same length
    plaintexts = [
        "ATTACKATDAWNBATTALIONONE",
        "SENDMORETROOPSTOFRONTLINE",
        "REPORTIMMEDIATELYONSTATUS",
        "HOLDTHEPOSITIONATALLCOST",
    ]
    rows, cols = 5, 5
    key_route = 'down_columns'
    
    # Encrypt all plaintexts with the same key
    ciphertexts = [route_cipher_encrypt(pt, rows, cols, key_route) for pt in plaintexts]
    print("Generated multiple ciphertexts with the same key.")

    # Create the transposed columns (the "depth")
    depth_cols = []
    for i in range(rows * cols):
        col = [ct[i] for ct in ciphertexts]
        depth_cols.append(col)
        
    # Analyze adjacency
    adjacency_scores = {}
    for i in range(len(depth_cols)):
        for j in range(len(depth_cols)):
            if i == j: continue
            # We only check horizontal adjacency for this demo
            score = score_adjacency(depth_cols[i], depth_cols[j])
            adjacency_scores[(i, j)] = score
            
    # Find the most likely adjacencies
    sorted_scores = sorted(adjacency_scores.items(), key=lambda item: item[1], reverse=True)
    
    print("\nTop 10 Most Likely HORIZONTAL Adjacencies (Ciphertext Pos1 -> Pos2):")
    for (pos1, pos2), score in sorted_scores[:10]:
        print(f"Pair ({pos1: >2}, {pos2: >2}) -> Score: {score:.2f}")

    print("\nANALYSIS: A 'down_columns' route means plaintexts are filled row by row.")
    print("So, grid cell (r,c) is adjacent to (r,c+1).")
    print("In the ciphertext, these correspond to positions (c*rows + r) and ((c+1)*rows + r).")
    print("For a 5x5 grid, pos 0 is adjacent to pos 5, pos 1 to 6, etc. See if the scores found this!")


######################################################################
# TECHNIQUE 3: MULTIPLE ANAGRAMMING (DEPTH ATTACK)
######################################################################
Generated multiple ciphertexts with the same key.

Top 10 Most Likely HORIZONTAL Adjacencies (Ciphertext Pos1 -> Pos2):
Pair (23,  4) -> Score: -14.90
Pair ( 0, 22) -> Score: -14.90
Pair (17, 22) -> Score: -15.00
Pair ( 5,  0) -> Score: -20.10
Pair (14, 19) -> Score: -20.10
Pair ( 6, 14) -> Score: -20.20
Pair (14, 10) -> Score: -20.20
Pair (21, 20) -> Score: -20.20
Pair (18,  4) -> Score: -20.20
Pair (18, 14) -> Score: -20.20

ANALYSIS: A 'down_columns' route means plaintexts are filled row by row.
So, grid cell (r,c) is adjacent to (r,c+1).
In the ciphertext, these correspond to positions (c*rows + r) and ((c+1)*rows + r).
For a 5x5 grid, pos 0 is adjacent to pos 5, pos 1 to 6, etc. See if the scores found this!


### Columnar transposition: encryption and decryption
Columnar transposition writes plaintext in rows under a keyword, then reads columns in the order determined by the keyword’s alphabetical order, with variants for padding and key repeats such as Myszkowski.
Double transposition applies two serial columnar steps and historically provided robust field security; a single step can often be attacked by testing grid heights and anagramming columns.



In [10]:
def columnar_key_to_order(key):
    # Produces numeric order 0..k-1 based on alphabetical order with stable ties
    key = key.upper()
    pairs = sorted([(ch, i) for i, ch in enumerate(key)])
    order = [None]*len(key)
    for rank, (_, original_idx) in enumerate(pairs):
        order[original_idx] = rank
    return order

def columnar_encrypt(pt, key, pad='X'):
    pt = normalize(pt)
    k = len(key)
    order = columnar_key_to_order(key)
    rows = math.ceil(len(pt)/k)
    grid = [['']*k for _ in range(rows)]
    idx = 0
    for r in range(rows):
        for c in range(k):
            if idx < len(pt):
                grid[r][c] = pt[idx]
                idx += 1
            else:
                grid[r][c] = pad
    out = []
    for c in sorted(range(k), key=lambda x: order[x]):
        for r in range(rows):
            out.append(grid[r][c])
    return ''.join(out)

def columnar_decrypt(ct, key):
    ct = normalize(ct)
    k = len(key)
    order = columnar_key_to_order(key)
    rows = math.ceil(len(ct)/k)
    # Determine column lengths for irregular fills (assume regular padding for simplicity here)
    col_len = rows
    grid = [['']*k for _ in range(rows)]
    idx = 0
    # Fill columns by order
    for c in sorted(range(k), key=lambda x: order[x]):
        for r in range(rows):
            grid[r][c] = ct[idx]
            idx += 1
    # Read rows
    pt = []
    for r in range(rows):
        for c in range(k):
            pt.append(grid[r][c])
    return ''.join(pt)


In [11]:
# use columnar
if __name__ == "__main__":
    plaintext = "WEAREDISCOVEREDFLEEATONCE"
    key = "ZEBRAS"
    ciphertext = columnar_encrypt(plaintext, key) # Encrypt the plaintext
    print(f"Columnar Cipher Ciphertext: {ciphertext}")
    decrypted_text = columnar_decrypt(ciphertext, key) # Decrypt the ciphertext
    print(f"Columnar Cipher Decrypted Text: {decrypted_text}")

Columnar Cipher Ciphertext: EVLNXACDTXESEAXROFOXDEECXWIREE
Columnar Cipher Decrypted Text: WEAREDISCOVEREDFLEEATONCEXXXXX


Open questions

Explain why factorial growth N! of column permutations quickly makes brute forcing impractical for long keys, and when small-key brute force and dictionary attacks are still useful in practice.
Think about how fast N! (N factorial) grows. Let's calculate the number of possible key arrangements (permutations) for different key lengths:

Key of length 4 (e.g., "LOCK"): 4! = 4 × 3 × 2 × 1 = 24 permutations. A computer can check this instantly.

Key of length 8 (e.g., "SECURITY"): 8! = 40,320 permutations. Still very easy for a computer.

Key of length 10 (e.g., "CIPHERTEXT"): 10! = 3,628,800 permutations. Manageable.

Key of length 14 (e.g., "TRANSPOSITION"): 14! ≈ 87 billion permutations. This is starting to get very time-consuming.

Key of length 20: 20! ≈ 2,432,902,008,176,640,000 (2.4 quintillion) permutations. This is computationally impossible to check.

For Brute-Force: What does this growth tell you about the maximum key length you could realistically try to crack by checking every single possible column order?

For Dictionary Attacks: Do humans usually pick random strings like "XGAYBJS" for keys, or do they pick words they can remember? How does this drastically reduce the number of keys you need to check compared to the full factorial number?

How does double transposition change the cryptanalytic workload and which detection strategies remain effective when multiple equal-length messages are available ?

The first encryption scrambles the columns. The second encryption takes that scrambled text and scrambles it again. What does this do to any remaining patterns? Does it make the ciphertext look more or less random?

### Scoring English: lightweight fitness function
State-of-the-art attacks often use quadgram statistics for fitness, ranking candidate plaintexts by log-likelihood under language models, which materially improves hill climbing performance.
For pedagogy and portability, the following lightweight score counts common bigrams, trigrams, vowels/consonants transitions, and basic word hits; it is weaker than quadgrams but sufficient to illustrate the methods and can be replaced later with quadgram scoring

Your above version of the program should be already loooking something like this, if not, take this as a new template and try to improve it.

Test it on the examples above to see how it performs

In [12]:
COMMON_BIGRAMS = set("""
TH HE IN ER AN RE ON AT EN ND TI ES OR TE OF ED IS IT AL AR ST TO NT NG SE HA AS OU IO LE IS OU
""".split())
COMMON_TRIGRAMS = set("THE AND ION TIO ENT ING HER FOR TER ERE".split())
COMMON_WORDS = set("THE AND THAT FOR WITH THIS HAVE FROM NOT BUT ALL ARE AS BE WAS WERE ONE".split())
COMMON_CRIBS = set("LL TT SS EE OO".split())

def english_score(s):
    s = s.upper()
    score = 0
    # Common bigrams
    for i in range(len(s)-1):
        bg = s[i:i+2]
        if bg in COMMON_BIGRAMS:
            score += 2
    # Common trigrams
    for i in range(len(s)-2):
        tg = s[i:i+3]
        if tg in COMMON_TRIGRAMS:
            score += 3
    # Vowel/consonant heuristic
    vowels = set("AEIOU")
    vc = sum((s[i] in vowels) ^ (s[i+1] in vowels) for i in range(len(s)-1))
    score += vc * 0.2
    # Word hits (space-insensitive approximate)
    chunk = ''.join(ch if ch in ALPHABET else ' ' for ch in s)
    tokens = chunk.split()
    score += sum(5 for t in tokens if t in COMMON_WORDS)
    return score


In [17]:
#use fitness function against columnar cipher encrypted text

if __name__ == "__main__":
    plaintext = "WEAREDISCOVEREDFLEEATONCE"
    key = "ZEBRAS"
    ciphertext = columnar_encrypt(plaintext, key) # Encrypt the plaintext
    print(f"Columnar Cipher Ciphertext: {ciphertext}")
    decrypted_text = columnar_decrypt(ciphertext, key) # Decrypt the ciphertext
    print(f"Columnar Cipher Decrypted Text: {decrypted_text}")

#fitness function against columnar cipher encrypted text
    fitness_score = english_score(ciphertext)
    print(f"Columnar Cipher Decrypted Text Fitness Score: {fitness_score}")

    fitness_score_plain = english_score(decrypted_text)
    print(f"Columnar Cipher Plaintext Fitness Score: {fitness_score_plain}")


# --

Columnar Cipher Ciphertext: EVLNXACDTXESEAXROFOXDEECXWIREE
Columnar Cipher Decrypted Text: WEAREDISCOVEREDFLEEATONCEXXXXX
Columnar Cipher Decrypted Text Fitness Score: 11.2
Columnar Cipher Plaintext Fitness Score: 28.6


Open questions

Replace the simple score with a quadgram log-likelihood model and compare convergence speed and accuracy in hill climbing across multiple restarts.

### Manual anagramming workflow for columnar transposition
Manual or semi-automated anagramming aligns the ciphertext into candidate grids, then reorders columns to expose probable words and letter patterns like QU and common suffixes, extending partial reconstructions iteratively.
This approach is particularly effective when there are multiple ciphertexts of the same length (depth), enabling simultaneous anagramming that reveals shared column orders.

In [18]:
def gridify_by_cols(ct, key_len):
    ct = normalize(ct)
    rows = math.ceil(len(ct)/key_len)
    cols = key_len
    grid = [['?']*cols for _ in range(rows)]
    idx = 0
    for c in range(cols):
        for r in range(rows):
            if idx < len(ct):
                grid[r][c] = ct[idx]
                idx += 1
    return grid  # columns filled left->right

def print_grid(grid):
    for r in grid:
        print(' '.join(r))

def permute_columns(grid, perm):
    rows = len(grid)
    cols = len(grid[0])
    new_grid = [['']*cols for _ in range(rows)]
    for new_c, old_c in enumerate(perm):
        for r in range(rows):
            new_grid[r][new_c] = grid[r][old_c]
    return new_grid

def read_rows(grid):
    return ''.join(ch for r in grid for ch in r)


In [19]:
cipher_example = columnar_encrypt("WEAREDISCOVEREDFLEEATONCE", key="ZEBRAS")
klen = 6
g = gridify_by_cols(cipher_example, klen)
print("Original columns:")
print_grid(g)
# Try a guess permutation (e.g., [4,2,1,3,0,5] for ZEBRAS order 6 3 2 4 1 5)
guess_perm = [4,2,1,3,0,5]
gg = permute_columns(g, guess_perm)
print("\nPermuted columns -> rows:")
print(read_rows(gg))


Original columns:
E A E R D W
V C S O E I
L D E F E R
N T A O C E
X X X X X E

Permuted columns -> rows:
DEAREWESCOVIEEDFLRCATONEXXXXXE


In [20]:
#use anagramming tools above to test columnar ciphertext
if __name__ == "__main__":
    print("\n" + "#" * 70)
    print("# TECHNIQUE 4: ANAGRAMMING COLUMNAR CIPHERTEXT")
    print("#" * 70)

    # --- Base Code for Students ---
    plaintext = "WEAREDISCOVEREDFLEEATONCE"
    key = "ZEBRAS"
    ciphertext = columnar_encrypt(plaintext, key) # Encrypt the plaintext
    print(f"Columnar Cipher Ciphertext: {ciphertext}")
    
    klen = len(key)
    grid = gridify_by_cols(ciphertext, klen)
    print("\nCiphertext arranged in columns:")
    print_grid(grid)
    
    # Example guess permutation (students can try different ones)
    guess_perm = [4,2,1,3,0,5]  # Corresponds to ZEBRAS order
    permuted_grid = permute_columns(grid, guess_perm)
    decrypted_attempt = read_rows(permuted_grid)
    
    print("\nDecrypted attempt with guessed column order:")
    print(decrypted_attempt)
    
    score = english_score(decrypted_attempt)
    print(f"Fitness score of this attempt: {score}")


######################################################################
# TECHNIQUE 4: ANAGRAMMING COLUMNAR CIPHERTEXT
######################################################################
Columnar Cipher Ciphertext: EVLNXACDTXESEAXROFOXDEECXWIREE

Ciphertext arranged in columns:
E A E R D W
V C S O E I
L D E F E R
N T A O C E
X X X X X E

Decrypted attempt with guessed column order:
DEAREWESCOVIEEDFLRCATONEXXXXXE
Fitness score of this attempt: 17.4


In [21]:
#brute force columnar cipher
def brute_force_columnar_attack(ct, max_key_len=10):
    for key_len in range(2, max_key_len + 1):
        grid = gridify_by_cols(ct, key_len)
        cols = list(range(key_len))
        best_score = -float('inf')
        best_plaintext = ""
        best_perm = None
        for perm in itertools.permutations(cols):
            permuted_grid = permute_columns(grid, perm)
            pt_candidate = read_rows(permuted_grid)
            score = english_score(pt_candidate)
            if score > best_score:
                best_score = score
                best_plaintext = pt_candidate
                best_perm = perm
        print(f"Key length {key_len}: Best score {best_score:.2f} with permutation {best_perm}")
        print(f"Decrypted plaintext: {best_plaintext}\n")
if __name__ == "__main__":
    print("\n" + "#" * 70)
    print("# TECHNIQUE 5: BRUTE-FORCE COLUMNAR ATTACK")
    print("#" * 70)

    # --- Base Code for Students ---
    plaintext = "WEAREDISCOVEREDFLEEATONCE"
    key = "ZEBRAS"
    ciphertext = columnar_encrypt(plaintext, key) # Encrypt the plaintext
    print(f"Columnar Cipher Ciphertext: {ciphertext}")
    
    brute_force_columnar_attack(ciphertext, max_key_len=6)


######################################################################
# TECHNIQUE 5: BRUTE-FORCE COLUMNAR ATTACK
######################################################################
Columnar Cipher Ciphertext: EVLNXACDTXESEAXROFOXDEECXWIREE
Key length 2: Best score 12.40 with permutation (1, 0)
Decrypted plaintext: REOVFLONXXDAECEDCTXXWEISREEAEX

Key length 3: Best score 15.40 with permutation (0, 2, 1)
Decrypted plaintext: EDEVESLEENCAXXXAWRCIODRFTEOXEX

Key length 4: Best score 15.20 with permutation (0, 1, 3, 2)
Decrypted plaintext: ETXOVXWFLEIONSRXXEEDAAEECX?EDR?C

Key length 5: Best score 17.80 with permutation (1, 4, 0, 3, 2)
Decrypted plaintext: CXEOEDWVXATILDXXRNEREEXEOSEACF

Key length 6: Best score 28.60 with permutation (5, 2, 1, 3, 0, 4)
Decrypted plaintext: WEAREDISCOVEREDFLEEATONCEXXXXX



Open questions

Design a heuristic that proposes column swaps when it forms likely digrams (e.g., matching Q with U) across row boundaries and evaluate its utility on short texts.

Remember anagramming can be manual, automatic or hybrid, but in any case, it is only meant for smaller ciphertexts.

Explain how simultaneous anagramming on multiple equal-length intercepts reveals column alignments and under what conditions it fails.

Use AI to get a functional code in action



### Hill climbing for columnar transposition
Hill climbing initializes a random column order for a given key length, applies local mutations (e.g., swaps, cuts, rotations), accepts improvements by a fitness function, and restarts multiple times to escape local maxima, which is a standard approach for columnar transposition.
Key length is often unknown; practical workflows try a range (e.g., 6–20), combining short-key enumeration, dictionary keys, and stochastic search to balance runtime and coverage.

python

In [22]:
def decrypt_with_perm(ct, perm):
    # perm is a permutation over columns [0..k-1] giving column read-out order
    ct = normalize(ct)
    k = len(perm)
    rows = math.ceil(len(ct)/k)
    # Fill columns in perm order
    grid = [['']*k for _ in range(rows)]
    idx = 0
    for c in perm:
        for r in range(rows):
            grid[r][c] = ct[idx]
            idx += 1
    # Read rows
    pt = []
    for r in range(rows):
        for c in range(k):
            pt.append(grid[r][c])
    return ''.join(pt)

def random_perm(k):
    p = list(range(k))
    random.shuffle(p)
    return p

def mutate_perm(p):
    p = p[:]
    choice = random.random()
    if choice < 0.5:
        # swap two positions
        i, j = random.sample(range(len(p)), 2)
        p[i], p[j] = p[j], p[i]
    elif choice < 0.75:
        # rotation
        r = random.randrange(1, len(p))
        p = p[r:] + p[:r]
    else:
        # cut and swap ends
        cut = random.randrange(1, len(p))
        p = p[cut:] + p[:cut]
    return p

def hill_climb_columnar(ct, k, iters=5000, restarts=50):
    best = None
    best_score = float('-inf')
    best_pt = None
    for _ in range(restarts):
        p = random_perm(k)
        pt = decrypt_with_perm(ct, p)
        s = english_score(pt)
        for _ in range(iters):
            q = mutate_perm(p)
            pt2 = decrypt_with_perm(ct, q)
            s2 = english_score(pt2)
            if s2 > s:
                p, s, pt = q, s2, pt2
        if s > best_score:
            best, best_score, best_pt = p[:], s, pt
    return best, best_score, best_pt


In [23]:
plaintext = "WHENINCRISISLOOKFORPATTERNSANDLETSTATISTICSGUIDEYOURSEARCH"
key = "ZEBRAS"
ct = columnar_encrypt(plaintext, key)
best_perm, best_score, best_pt = hill_climb_columnar(ct, k=len(key), iters=4000, restarts=60)
print("Ciphertext:", ct)
print("Recovered (no spacing):", best_pt)
print("Best perm:", best_perm, "Score:", best_score)

Ciphertext: IIFTNTIDSXEIOASTSUUCHROPNEIGORNSKTASTIRHNSOEDACEEXWCLRRLTSYA
Recovered (no spacing): WHENINCRISISLOOKFORPATTERNSANDLETSTATISTICSGUIDEYOURSEARCHXX
Best perm: [4, 2, 1, 3, 5, 0] Score: 57.8


### Short-key enumeration and dictionary keys
Testing all permutations up to about length 9 is feasible since 
N
!
N! remains small enough, after which dictionary-key trials and hill climbing become more appropriate due to combinatorial explosion.
A practical pipeline tries factorial enumeration for small k, then dictionary words for medium k, then stochastic search over 10–20, with multiple restarts and early stopping after score plateaus

In [24]:
def brute_force_small_k(ct, k, max_perms=500000):
    ct = normalize(ct)
    best = None
    best_score = float('-inf')
    best_pt = None
    count = 0
    for perm in itertools.permutations(range(k)):
        pt = decrypt_with_perm(ct, list(perm))
        s = english_score(pt)
        if s > best_score:
            best, best_score, best_pt = list(perm), s, pt
        count += 1
        if count >= max_perms:
            break
    return best, best_score, best_pt

# Example: DO NOT run for k>9 in class; illustrate on tiny examples only


In [25]:
#example of brute force small k 8
if __name__ == "__main__":
    print("\n" + "#" * 70)
    print("# TECHNIQUE 6: BRUTE-FORCE SMALL K")
    print("#" * 70)

    # --- Base Code for Students ---
    plaintext = "WEAREDISCOVEREDFLEEATONCETHISTIMEISCRUCIALDONOTUSETHEBRIDGENORTHEBOAT"
    key = "BRILLIANT"
    ciphertext = columnar_encrypt(plaintext, key) # Encrypt the plaintext
    print(f"Columnar Cipher Ciphertext: {ciphertext}")
    
    klen = len(key)
    best_perm, best_score, best_pt = brute_force_small_k(ciphertext, k=klen, max_perms=500000)
    
    print(f"Best permutation found: {best_perm}")
    print(f"Best score: {best_score:.2f}")
    print(f"Decrypted plaintext: {best_pt}")


######################################################################
# TECHNIQUE 6: BRUTE-FORCE SMALL K
######################################################################
Columnar Cipher Ciphertext: IFEIDHOXWOEIRORHAETTCUDBDDCELTNTRROIISGOEENMAEEASLTSOERXEVASUTIECEHCNBTX
Best permutation found: [7, 0, 1, 4, 2, 3, 8, 6, 5]
Best score: 77.80
Decrypted plaintext: WAREDCEISOEREDEVFLETONCHAETITIMECSISRCIALNUDOOUSETBTHERDGENTIORHBOATXEXX


Test on bigger ciphertexts, establish a benchmark table where you can see this attacks limitations in computational power with your laptop

can you think of a deep learning approach that can combine both heuristics hill climb and transformer attention mechanisms?   